In [1]:
!pip install -q -U transformers accelerate sentencepiece safetensors
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 97.4 MB/s eta 0:00:00
True Tesla T4


In [2]:
!cp -r /kaggle/input/datasets/rifatbinreza/alta2026-pipeline/alta2026_pipeline /kaggle/working/
%cd /kaggle/working/alta2026_pipeline
!ls

/kaggle/working/alta2026_pipeline
answer_initial.csv	  infer.py	    thresholds.py
artifacts		  metadata	    train.csv
blend.py		  metrics.py	    transformer_multitask.py
classical_baseline.py	  README.md	    validate_answer.py
classical_baseline_v2.py  requirements.txt  valid.csv
evaluate.py		  run_baseline.sh


In [3]:
%%writefile idan_model.py
"""
idan_model.py — Incongruity-Aware Dual-Attention Network (IDAN)

Novel architecture (not a fine-tuned-transformer-only baseline):

  1. LITERAL BRANCH: multi-kernel 1D CNN over token embeddings, capturing
     local/surface-level sentiment cues (what the sentence "appears" to say).
  2. CONTEXTUAL BRANCH: pretrained transformer encoder (DeBERTa), capturing
     the sentence's actual discourse-level/contextual meaning.
  3. INCONGRUITY FUSION: the two branches are combined into a joint
     [literal ; contextual] representation, per token, then passed through
     a CBAM-style (Woo et al. 2018) dual attention module:
       - Channel attention: which FEATURE DIMENSIONS (of literal vs
         contextual signal) matter for this example.
       - Spatial attention: which TOKEN POSITIONS carry the strongest
         mismatch signal between literal and contextual meaning.
     We additionally compute an explicit incongruity vector
     (elementwise difference and product between literal- and
     contextual-branch pooled representations), following the standard
     NLI-style mismatch-feature trick, giving the model a direct numeric
     signal for "these two views of the sentence disagree."
  4. Fused representation -> two classification heads (sentiment, sarcasm),
     same multi-task setup as transformer_multitask.py, same source/variety
     control-token conditioning.

This is intentionally NOT a from-scratch pretrained language model — that
would require far more data/compute than is available here and would
likely underperform a fine-tuned encoder. The novelty is the attention/
fusion mechanism built on top of existing pretrained representations,
consistent with how recent incongruity-based sarcasm papers (ACL/COLING
2025) structure their contributions.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


class LiteralCNNBranch(nn.Module):
    """Multi-kernel 1D CNN over token embeddings -> per-token local feature map.
    Uses 'same' padding so output sequence length matches input length,
    which lets us align this branch token-for-token with the transformer's
    hidden states downstream.
    """
    def __init__(self, embed_dim: int, out_channels: int = 256, kernel_sizes=(2, 3, 4, 5)):
        super().__init__()
        per_k = out_channels // len(kernel_sizes)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, per_k, kernel_size=k, padding=k // 2)
            for k in kernel_sizes
        ])
        self.out_channels = per_k * len(kernel_sizes)
        self.act = nn.GELU()
        self.norm = nn.LayerNorm(self.out_channels)

    def forward(self, token_embeds: torch.Tensor) -> torch.Tensor:
        # token_embeds: (B, L, E) -> conv wants (B, E, L)
        x = token_embeds.transpose(1, 2)
        feats = []
        for conv in self.convs:
            f = self.act(conv(x))
            # conv with even kernel + this padding can produce L+1; trim to L
            f = f[:, :, :token_embeds.size(1)]
            feats.append(f)
        out = torch.cat(feats, dim=1)          # (B, C, L)
        out = out.transpose(1, 2)               # (B, L, C)
        return self.norm(out)


class LiteralLinearBranch(nn.Module):
    """ABLATION variant of the literal branch: a per-token linear projection
    of the raw embeddings with NO convolution -- i.e. no local n-gram
    context at all, just a reshaped bag-of-embeddings. Used to test whether
    the CNN's local pattern-matching is actually doing work, or whether any
    projected view of the raw embeddings would fuse similarly."""
    def __init__(self, embed_dim: int, out_channels: int = 256):
        super().__init__()
        self.proj = nn.Linear(embed_dim, out_channels)
        self.act = nn.GELU()
        self.norm = nn.LayerNorm(out_channels)
        self.out_channels = out_channels

    def forward(self, token_embeds: torch.Tensor) -> torch.Tensor:
        return self.norm(self.act(self.proj(token_embeds)))


class ChannelAttention(nn.Module):
    """CBAM-style channel attention: squeeze spatial (sequence) dimension via
    both avg- and max-pooling, pass through a shared MLP, sum, sigmoid.
    Tells the model which FEATURE CHANNELS (literal-branch dims vs
    contextual-branch dims) matter most for this example.

    reduction is raised to 16 (from the original CBAM paper's 16, not 8) and
    dropout is added inside the MLP -- on a dataset this small (~1.8k
    examples/fold), the extra squeeze-excite parameters are a real
    overfitting risk, so keep this module as lightweight as possible."""
    def __init__(self, channels: int, reduction: int = 16, dropout: float = 0.1):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.mlp = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, channels),
        )

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # x: (B, L, C), mask: (B, L) with 1 for real tokens, 0 for padding
        m = mask.unsqueeze(-1)                          # (B, L, 1)
        denom = m.sum(dim=1).clamp(min=1.0)
        avg_pool = (x * m).sum(dim=1) / denom            # (B, C)
        max_pool = (x.masked_fill(m == 0, float("-inf"))).max(dim=1).values
        channel_att = torch.sigmoid(self.mlp(avg_pool) + self.mlp(max_pool))  # (B, C)
        return x * channel_att.unsqueeze(1)              # (B, L, C)


class SpatialAttention(nn.Module):
    """CBAM-style spatial attention: squeeze channel dimension via avg- and
    max-pooling, concat, 1D conv over the sequence, sigmoid.
    Tells the model which TOKEN POSITIONS carry the strongest
    literal-vs-contextual mismatch (the actual incongruity signal)."""
    def __init__(self, kernel_size: int = 5):
        super().__init__()
        self.conv = nn.Conv1d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # x: (B, L, C)
        avg_pool = x.mean(dim=2, keepdim=True)           # (B, L, 1)
        max_pool = x.max(dim=2, keepdim=True).values     # (B, L, 1)
        pooled = torch.cat([avg_pool, max_pool], dim=2).transpose(1, 2)  # (B, 2, L)
        att = torch.sigmoid(self.conv(pooled)).transpose(1, 2)           # (B, L, 1)
        att = att * mask.unsqueeze(-1)
        return x * att


class IncongruityDualAttention(nn.Module):
    """Fuses literal + contextual token representations, applies channel
    then spatial attention (CBAM ordering), and computes an explicit
    incongruity vector from the pooled branch representations.

    Ablation switches:
      use_channel_attention / use_spatial_attention: turn off either half
        of the CBAM-style module (both default True = full IDAN).
      use_incongruity_features: turn off the explicit diff/product mismatch
        vector, keeping only the attention-fused pooled representation.
    """
    def __init__(self, literal_dim: int, contextual_dim: int, proj_dim: int = 384,
                 use_channel_attention: bool = True, use_spatial_attention: bool = True,
                 use_incongruity_features: bool = True):
        super().__init__()
        self.lit_proj = nn.Linear(literal_dim, proj_dim)
        self.ctx_proj = nn.Linear(contextual_dim, proj_dim)
        joint_dim = proj_dim * 2
        self.use_channel_attention = use_channel_attention
        self.use_spatial_attention = use_spatial_attention
        self.use_incongruity_features = use_incongruity_features
        if use_channel_attention:
            self.channel_att = ChannelAttention(joint_dim)
        if use_spatial_attention:
            self.spatial_att = SpatialAttention()
        self.fuse_norm = nn.LayerNorm(joint_dim)
        self.proj_dim = proj_dim

    def output_dim(self) -> int:
        # pooled_fused is always proj_dim*2; diff+prod each add proj_dim if enabled
        return self.proj_dim * 2 + (self.proj_dim * 2 if self.use_incongruity_features else 0)

    def forward(self, literal_feats, contextual_feats, mask):
        lit = self.lit_proj(literal_feats)     # (B, L, P)
        ctx = self.ctx_proj(contextual_feats)  # (B, L, P)
        joint = torch.cat([lit, ctx], dim=-1)  # (B, L, 2P)

        if self.use_channel_attention:
            joint = self.channel_att(joint, mask)
        if self.use_spatial_attention:
            joint = self.spatial_att(joint, mask)
        joint = self.fuse_norm(joint)

        # masked mean pooling over sequence for the fused representation
        m = mask.unsqueeze(-1)
        denom = m.sum(dim=1).clamp(min=1.0)
        pooled_fused = (joint * m).sum(dim=1) / denom          # (B, 2P)

        if not self.use_incongruity_features:
            return pooled_fused

        # explicit incongruity features: literal vs contextual mismatch,
        # computed on the pooled (sentence-level) branch representations
        lit_pooled = (lit * m).sum(dim=1) / denom              # (B, P)
        ctx_pooled = (ctx * m).sum(dim=1) / denom              # (B, P)
        diff = lit_pooled - ctx_pooled
        prod = lit_pooled * ctx_pooled

        return torch.cat([pooled_fused, diff, prod], dim=-1)   # (B, 2P + P + P) = (B, 4P)


class SubgroupExpertGate(nn.Module):
    """Novel contribution #2: subgroup-conditioned mixture-of-experts gating.

    Motivation: the ALTA scoring formula explicitly punishes the WORST
    dialect per task, and sarcasm rates swing from ~0.06% (en-UK/Google) to
    ~42% (en-AU/Reddit) across source x variety subgroups. A single shared
    transformation (even with control-token conditioning, which only lets
    the model *see* subgroup identity) may struggle to represent such
    divergent subgroup-specific decision boundaries equally well. This
    module explicitly ROUTES each example through a soft mixture of K
    expert transformations, conditioned on its (source, variety) identity,
    so the model can allocate specialized capacity to underperforming
    subgroups rather than sharing one transformation across all of them.

    To our knowledge, subgroup-conditioned expert routing has not been
    applied to dialect-robust sarcasm/sentiment classification under a
    worst-case (min-based) evaluation metric -- this directly targets what
    the competition metric measures, rather than average-case performance.
    """
    def __init__(self, input_dim: int, n_subgroups: int = 4, n_experts: int = 3,
                 expert_hidden: int = 256, dropout: float = 0.1):
        super().__init__()
        self.subgroup_embed = nn.Embedding(n_subgroups, expert_hidden)
        self.gate = nn.Sequential(
            nn.Linear(expert_hidden, expert_hidden),
            nn.GELU(),
            nn.Linear(expert_hidden, n_experts),
        )
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, input_dim),
                nn.GELU(),
                nn.Dropout(dropout),
            )
            for _ in range(n_experts)
        ])
        self.n_experts = n_experts

    def forward(self, fused: torch.Tensor, subgroup_id: torch.Tensor) -> torch.Tensor:
        # fused: (B, D), subgroup_id: (B,) long tensor
        sg_emb = self.subgroup_embed(subgroup_id)
        gate_logits = self.gate(sg_emb)
        gate_weights = torch.softmax(gate_logits, dim=-1)

        expert_outs = torch.stack([e(fused) for e in self.experts], dim=1)
        mixed = (expert_outs * gate_weights.unsqueeze(-1)).sum(dim=1)
        return fused + mixed  # residual: experts refine, don't replace, the fused representation


class IDAN(nn.Module):
    """Full model: transformer encoder (contextual branch) + CNN (literal
    branch) + incongruity dual-attention fusion + two task heads.

    Ablation switches (all default to the full IDAN configuration):
      literal_encoder: 'cnn' (default) or 'linear' -- 'linear' removes the
        multi-kernel CNN's local n-gram modeling, testing whether the CNN
        itself matters vs. any projected view of raw embeddings.
      use_channel_attention / use_spatial_attention: disable either half
        of the CBAM-style fusion module.
      use_incongruity_features: disable the explicit diff/product mismatch
        vector, keeping only the attention-fused pooled representation.
    """
    def __init__(self, encoder, hidden_size: int, cnn_out_channels: int = 256,
                 proj_dim: int = 384, dropout: float = 0.15,
                 literal_encoder: str = "cnn",
                 use_channel_attention: bool = True,
                 use_spatial_attention: bool = True,
                 use_incongruity_features: bool = True,
                 use_subgroup_gate: bool = False,
                 n_subgroups: int = 4,
                 n_experts: int = 3):
        super().__init__()
        self.encoder = encoder  # a pretrained AutoModel, e.g. DeBERTa
        self.embed_layer = encoder.get_input_embeddings()

        if literal_encoder == "cnn":
            self.literal_branch = LiteralCNNBranch(
                embed_dim=self.embed_layer.embedding_dim, out_channels=cnn_out_channels
            )
        elif literal_encoder == "linear":
            self.literal_branch = LiteralLinearBranch(
                embed_dim=self.embed_layer.embedding_dim, out_channels=cnn_out_channels
            )
        else:
            raise ValueError(f"unknown literal_encoder: {literal_encoder}")

        self.fusion = IncongruityDualAttention(
            literal_dim=self.literal_branch.out_channels,
            contextual_dim=hidden_size,
            proj_dim=proj_dim,
            use_channel_attention=use_channel_attention,
            use_spatial_attention=use_spatial_attention,
            use_incongruity_features=use_incongruity_features,
        )
        fused_dim = self.fusion.output_dim()

        self.use_subgroup_gate = use_subgroup_gate
        if use_subgroup_gate:
            self.subgroup_gate = SubgroupExpertGate(
                input_dim=fused_dim, n_subgroups=n_subgroups, n_experts=n_experts, dropout=dropout
            )

        self.drop = nn.Dropout(dropout)
        self.sent_head = nn.Sequential(nn.Linear(fused_dim, proj_dim), nn.GELU(), nn.Linear(proj_dim, 2))
        self.sarc_head = nn.Sequential(nn.Linear(fused_dim, proj_dim), nn.GELU(), nn.Linear(proj_dim, 2))

    def forward(self, input_ids, attention_mask, token_type_ids=None, subgroup_id=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        enc_out = self.encoder(**kwargs)
        contextual_feats = enc_out.last_hidden_state       # (B, L, H) -- contextual branch

        token_embeds = self.embed_layer(input_ids)         # (B, L, E)
        literal_feats = self.literal_branch(token_embeds)  # (B, L, C) -- literal branch

        mask = attention_mask.float()
        fused = self.fusion(literal_feats, contextual_feats, mask)  # (B, fused_dim)

        if self.use_subgroup_gate:
            if subgroup_id is None:
                raise ValueError("use_subgroup_gate=True but no subgroup_id was passed to forward()")
            fused = self.subgroup_gate(fused, subgroup_id)

        fused = self.drop(fused)
        return self.sent_head(fused), self.sarc_head(fused)

Writing idan_model.py


In [4]:
%%writefile idan_train.py
"""
idan_train.py — trains IDAN (Incongruity-Aware Dual-Attention Network),
optionally with the SubgroupExpertGate (subgroup-conditioned MoE routing).
"""
from __future__ import annotations

import argparse
import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

from metrics import alta_score, competition_stratify_key
from transformer_multitask import SPECIAL_TOKENS, TextDataset
from idan_model import IDAN


# Fixed (source, variety) -> integer subgroup id mapping. Confirmed against
# the actual data: exactly these 4 combinations exist, no others.
SUBGROUP_MAP = {
    ("google", "en-AU"): 0,
    ("reddit", "en-AU"): 1,
    ("google", "en-UK"): 2,
    ("reddit", "en-UK"): 3,
}
N_SUBGROUPS = len(SUBGROUP_MAP)
SUBGROUP_LABELS = {v: f"{src}/{var}" for (src, var), v in SUBGROUP_MAP.items()}


def subgroup_ids_for(df: pd.DataFrame) -> np.ndarray:
    ids = df.apply(lambda r: SUBGROUP_MAP[(r["source"], r["variety"])], axis=1)
    return ids.to_numpy(dtype=np.int64)


class IDANDataset(TextDataset):
    """Extends the shared TextDataset with a subgroup_id field, needed only
    when the SubgroupExpertGate is enabled. Reuses all of TextDataset's
    tokenization/control-token logic unchanged."""
    def __init__(self, df, tokenizer, max_length, with_labels=True):
        super().__init__(df, tokenizer, max_length, with_labels)
        self.subgroup_ids = subgroup_ids_for(self.df)

    def __getitem__(self, i):
        item = super().__getitem__(i)
        item["subgroup_id"] = torch.tensor(self.subgroup_ids[i], dtype=torch.long)
        return item


@dataclass
class IDANConfig:
    backbone: str = "microsoft/deberta-v3-base"
    max_length: int = 192
    batch_size: int = 16
    epochs: int = 10
    lr: float = 2e-5
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    dropout: float = 0.15
    cnn_out_channels: int = 256
    proj_dim: int = 384
    sentiment_loss_weight: float = 1.0
    sarcasm_loss_weight: float = 1.25
    gradient_accumulation_steps: int = 2
    seed: int = 42
    n_folds: int = 3
    early_stopping_patience: int = 2
    literal_encoder: str = "cnn"
    use_channel_attention: bool = True
    use_spatial_attention: bool = True
    use_incongruity_features: bool = True
    max_sarcasm_class_weight: float = 5.0
    use_subgroup_gate: bool = False
    n_experts: int = 3


def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def build_model(cfg: IDANConfig, n_new_tokens: int):
    encoder = AutoModel.from_pretrained(cfg.backbone, torch_dtype=torch.float32)
    encoder.resize_token_embeddings(encoder.config.vocab_size + n_new_tokens)
    model = IDAN(
        encoder=encoder,
        hidden_size=encoder.config.hidden_size,
        cnn_out_channels=cfg.cnn_out_channels,
        proj_dim=cfg.proj_dim,
        dropout=cfg.dropout,
        literal_encoder=cfg.literal_encoder,
        use_channel_attention=cfg.use_channel_attention,
        use_spatial_attention=cfg.use_spatial_attention,
        use_incongruity_features=cfg.use_incongruity_features,
        use_subgroup_gate=cfg.use_subgroup_gate,
        n_subgroups=N_SUBGROUPS,
        n_experts=cfg.n_experts,
    )
    return model


def _forward(model, cfg, batch, device):
    ids = batch["input_ids"].to(device)
    mask = batch["attention_mask"].to(device)
    tt = batch.get("token_type_ids")
    if tt is not None: tt = tt.to(device)
    sg = batch["subgroup_id"].to(device) if cfg.use_subgroup_gate else None
    return model(ids, mask, tt, subgroup_id=sg)


def evaluate(model, cfg, loader, device):
    model.eval(); ps, pz = [], []
    with torch.no_grad():
        for batch in loader:
            a, b = _forward(model, cfg, batch, device)
            ps.append(torch.softmax(a, -1)[:, 1].cpu().numpy())
            pz.append(torch.softmax(b, -1)[:, 1].cpu().numpy())
    return np.concatenate(ps), np.concatenate(pz)


def log_subgroup_expert_weights(model, device, fold, out_dir):
    """Diagnostic: the SubgroupExpertGate's routing weights depend ONLY on
    subgroup identity (not on individual example content), so we can read
    off exactly what each subgroup has learned to prefer by feeding all 4
    subgroup ids through the gate directly -- no inference pass needed."""
    if not model.use_subgroup_gate:
        return None
    model.eval()
    with torch.no_grad():
        all_ids = torch.arange(N_SUBGROUPS, device=device)
        sg_emb = model.subgroup_gate.subgroup_embed(all_ids)
        gate_logits = model.subgroup_gate.gate(sg_emb)
        gate_weights = torch.softmax(gate_logits, dim=-1).cpu().numpy()

    table = {}
    for sg_id in range(N_SUBGROUPS):
        label = SUBGROUP_LABELS[sg_id]
        table[label] = {f"expert_{k}": float(gate_weights[sg_id, k]) for k in range(gate_weights.shape[1])}

    print(f"[IDAN] fold={fold}: subgroup -> expert routing weights:")
    for label, weights in table.items():
        weights_str = ", ".join(f"{k}={v:.3f}" for k, v in weights.items())
        print(f"    {label}: {weights_str}")

    with open(Path(out_dir) / f"fold{fold}_subgroup_weights.json", "w") as f:
        json.dump(table, f, indent=2)
    return table


def train_one_fold(train_df, val_df, cfg, out_dir, fold):
    set_seed(cfg.seed + fold)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(cfg.backbone, use_fast=True)
    tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})

    model = build_model(cfg, n_new_tokens=len(SPECIAL_TOKENS)).to(device)

    tr_ds = IDANDataset(train_df, tokenizer, cfg.max_length, True)
    va_ds = IDANDataset(val_df, tokenizer, cfg.max_length, True)
    tr_loader = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True)
    va_loader = DataLoader(va_ds, batch_size=cfg.batch_size * 2, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    steps = max(1, len(tr_loader) * cfg.epochs // cfg.gradient_accumulation_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(steps * cfg.warmup_ratio), steps)

    s_loss = nn.CrossEntropyLoss()
    n_pos = max(int(train_df["sarcasm"].sum()), 1)
    n_neg = max(len(train_df) - n_pos, 1)
    pos_weight = min(n_neg / n_pos, cfg.max_sarcasm_class_weight)
    z_weight = torch.tensor([1.0, pos_weight], device=device)
    z_loss = nn.CrossEntropyLoss(weight=z_weight)
    print(f"[IDAN] fold={fold}: sarcasm class weight = {pos_weight:.2f} (n_pos={n_pos}, n_neg={n_neg}), subgroup_gate={cfg.use_subgroup_gate}")
    best, best_state, epochs_since_improve = -1.0, None, 0

    swa_state = None
    swa_count = 0
    SWA_TOLERANCE = 0.01

    for epoch in range(cfg.epochs):
        model.train(); optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(tr_loader):
            a, b = _forward(model, cfg, batch, device)
            loss = (cfg.sentiment_loss_weight * s_loss(a, batch["sentiment"].to(device))
                    + cfg.sarcasm_loss_weight * z_loss(b, batch["sarcasm"].to(device)))
            loss = loss / cfg.gradient_accumulation_steps
            loss.backward()
            if (step + 1) % cfg.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)

        ps, pz = evaluate(model, cfg, va_loader, device)
        tmp = val_df[["source", "variety", "text"]].copy()
        tmp["sentiment"] = (ps >= 0.5).astype(int)
        tmp["sarcasm"] = (pz >= 0.5).astype(int)
        scores, comp = alta_score(val_df, tmp)
        print(f"[IDAN] fold={fold} epoch={epoch+1} score={comp:.5f} {scores}")

        if comp > best:
            best = comp
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_since_improve = 0
        else:
            epochs_since_improve += 1

        if comp >= best - SWA_TOLERANCE:
            current_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if swa_state is None:
                swa_state = current_state
                swa_count = 1
            else:
                swa_count += 1
                for k in swa_state:
                    if swa_state[k].dtype.is_floating_point:
                        swa_state[k] = swa_state[k] + (current_state[k] - swa_state[k]) / swa_count

        if epochs_since_improve >= cfg.early_stopping_patience:
            print(f"[IDAN] fold={fold}: no improvement for {cfg.early_stopping_patience} epochs, stopping early at epoch {epoch+1}")
            break

    if swa_state is not None and swa_count > 1:
        model.load_state_dict(swa_state)
        ps, pz = evaluate(model, cfg, va_loader, device)
        tmp = val_df[["source", "variety", "text"]].copy()
        tmp["sentiment"] = (ps >= 0.5).astype(int)
        tmp["sarcasm"] = (pz >= 0.5).astype(int)
        _, swa_comp = alta_score(val_df, tmp)
        print(f"[IDAN] fold={fold}: SWA (n={swa_count} epochs) score={swa_comp:.5f} vs best-single-epoch={best:.5f}")
        if swa_comp > best:
            print(f"[IDAN] fold={fold}: SWA wins, using averaged weights")
            best = swa_comp
            best_state = swa_state
        else:
            model.load_state_dict(best_state)

    model.load_state_dict(best_state)
    log_subgroup_expert_weights(model, device, fold, out_dir)
    torch.save(model.state_dict(), out_dir / f"fold{fold}.pt")
    with open(out_dir / f"fold{fold}.json", "w") as f:
        json.dump({"fold": fold, "best_validation_score": best, "config": asdict(cfg)}, f, indent=2)
    return model, tokenizer, device


def cv_train(train_csv, out_dir, cfg: IDANConfig, external_valid_csv=None):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(train_csv)
    keys = competition_stratify_key(df)
    skf = StratifiedKFold(cfg.n_folds, shuffle=True, random_state=cfg.seed)

    ext = pd.read_csv(external_valid_csv) if external_valid_csv else None
    ext_ps_all, ext_pz_all = [], []
    oof = df[["source", "variety", "text", "sentiment", "sarcasm"]].copy()
    oof["p_sentiment"] = np.nan; oof["p_sarcasm"] = np.nan

    for fold, (tr, va) in enumerate(skf.split(df, keys)):
        model, tokenizer, device = train_one_fold(df.iloc[tr].copy(), df.iloc[va].copy(), cfg, out, fold)

        va_ds = IDANDataset(df.iloc[va].copy(), tokenizer, cfg.max_length, True)
        va_loader = DataLoader(va_ds, batch_size=cfg.batch_size * 2, shuffle=False)
        ps, pz = evaluate(model, cfg, va_loader, device)
        oof.loc[df.index[va], "p_sentiment"] = ps
        oof.loc[df.index[va], "p_sarcasm"] = pz

        if ext is not None:
            ext_ds = IDANDataset(ext.copy(), tokenizer, cfg.max_length, False)
            ext_loader = DataLoader(ext_ds, batch_size=cfg.batch_size * 2, shuffle=False)
            eps, epz = evaluate(model, cfg, ext_loader, device)
            ext_ps_all.append(eps); ext_pz_all.append(epz)

    oof.to_csv(out / "oof_probabilities.csv", index=False)
    pred = oof[["source", "variety", "text"]].copy()
    pred["sentiment"] = (oof.p_sentiment >= .5).astype(int)
    pred["sarcasm"] = (oof.p_sarcasm >= .5).astype(int)
    scores, comp = alta_score(df, pred)
    print("[IDAN] OOF @0.5", scores, comp)

    if ext is not None:
        ext_probs = ext[["source", "variety", "text"]].copy()
        ext_probs["p_sentiment"] = np.mean(ext_ps_all, axis=0)
        ext_probs["p_sarcasm"] = np.mean(ext_pz_all, axis=0)
        ext_probs.to_csv(out / "valid_probabilities.csv", index=False)
        ext_pred = ext_probs[["source", "variety", "text"]].copy()
        ext_pred["sentiment"] = (ext_probs.p_sentiment >= .5).astype(int)
        ext_pred["sarcasm"] = (ext_probs.p_sarcasm >= .5).astype(int)
        ext_scores, ext_comp = alta_score(ext, ext_pred)
        print("[IDAN] External valid @0.5", ext_scores, ext_comp)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--train", default="train.csv")
    ap.add_argument("--valid", default=None)
    ap.add_argument("--out", default="artifacts/idan")
    ap.add_argument("--backbone", default="microsoft/deberta-v3-base")
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--max-length", type=int, default=192)
    ap.add_argument("--n-folds", type=int, default=3)
    ap.add_argument("--patience", type=int, default=2)
    ap.add_argument("--grad-accum", type=int, default=2)
    ap.add_argument("--literal-encoder", choices=["cnn", "linear"], default="cnn")
    ap.add_argument("--no-channel-attention", action="store_true")
    ap.add_argument("--no-spatial-attention", action="store_true")
    ap.add_argument("--no-incongruity-features", action="store_true")
    ap.add_argument("--no-class-weight", action="store_true")
    ap.add_argument("--use-subgroup-gate", action="store_true", help="enable subgroup-conditioned mixture-of-experts gating")
    ap.add_argument("--n-experts", type=int, default=3)
    ap.add_argument("--seed", type=int, default=42, help="random seed, for multi-seed variance estimation")
    args, _unknown = ap.parse_known_args()

    cfg = IDANConfig(
        backbone=args.backbone, epochs=args.epochs, batch_size=args.batch_size,
        max_length=args.max_length, n_folds=args.n_folds,
        early_stopping_patience=args.patience, gradient_accumulation_steps=args.grad_accum,
        literal_encoder=args.literal_encoder,
        use_channel_attention=not args.no_channel_attention,
        use_spatial_attention=not args.no_spatial_attention,
        use_incongruity_features=not args.no_incongruity_features,
        max_sarcasm_class_weight=1.0 if args.no_class_weight else 5.0,
        use_subgroup_gate=args.use_subgroup_gate,
        n_experts=args.n_experts,
        seed=args.seed,
    )
    cv_train(args.train, args.out, cfg, args.valid)

Writing idan_train.py


In [5]:
!python idan_train.py \
  --train train.csv \
  --valid valid.csv \
  --out artifacts/idan_gate_seed42 \
  --backbone microsoft/deberta-v3-base \
  --epochs 10 \
  --batch-size 16 \
  --max-length 192 \
  --n-folds 3 \
  --patience 2 \
  --grad-accum 2 \
  --no-class-weight \
  --use-subgroup-gate \
  --n-experts 3 \
  --seed 42

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
config.json: 100%|█████████████████████████████| 579/579 [00:00<00:00, 3.12MB/s]
tokenizer_config.json: 100%|██████████████████| 52.0/52.0 [00:00<00:00, 301kB/s]
spm.model: 100%|███████████████████████████| 2.46M/2.46M [00:00<00:00, 4.01MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
pytorch_model.bin: 100%|██████████████████████| 371M/371M [00:02<00:00, 132MB/s]
Loading weights: 100%|███████████████████████| 198/198 [00:00<00:00, 710.96it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.classifier.b

In [6]:
!python idan_train.py \
  --train train.csv \
  --valid valid.csv \
  --out artifacts/idan_gate_seed123 \
  --backbone microsoft/deberta-v3-base \
  --epochs 10 \
  --batch-size 16 \
  --max-length 192 \
  --n-folds 3 \
  --patience 2 \
  --grad-accum 2 \
  --no-class-weight \
  --use-subgroup-gate \
  --n-experts 3 \
  --seed 123

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████| 198/198 [00:00<00:00, 1183.47it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

In [7]:
!python idan_train.py \
  --train train.csv \
  --valid valid.csv \
  --out artifacts/idan_gate_seed2024 \
  --backbone microsoft/deberta-v3-base \
  --epochs 10 \
  --batch-size 16 \
  --max-length 192 \
  --n-folds 3 \
  --patience 2 \
  --grad-accum 2 \
  --no-class-weight \
  --use-subgroup-gate \
  --n-experts 3 \
  --seed 2024

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████| 198/198 [00:00<00:00, 1022.63it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

In [8]:
%%writefile aggregate_seeds.py
"""
aggregate_seeds.py — after running the same config with multiple seeds,
this loads each run's calibrated valid_probabilities.csv, applies its own
tuned thresholds, computes the ALTA score, and reports mean +/- std across
seeds.
"""
import json
import sys
import numpy as np
import pandas as pd
from thresholds import optimize_thresholds, apply_thresholds
from metrics import alta_score

valid = pd.read_csv("valid.csv")

run_dirs = [
    "artifacts/idan_gate_seed42",
    "artifacts/idan_gate_seed123",
    "artifacts/idan_gate_seed2024",
]

all_scores = []
all_components = []
for d in run_dirs:
    probs = pd.read_csv(f"{d}/valid_probabilities.csv")
    th = optimize_thresholds(valid, probs)
    pred = apply_thresholds(probs, th)
    scores, final = alta_score(valid, pred)
    print(f"{d}: {scores} -> ALTA={final:.4f}")
    all_scores.append(final)
    all_components.append(scores)

all_scores = np.array(all_scores)
print("\n=== Summary across seeds ===")
print(f"ALTA score: mean={all_scores.mean():.4f}  std={all_scores.std(ddof=1):.4f}  "
      f"min={all_scores.min():.4f}  max={all_scores.max():.4f}")

for key in all_components[0].keys():
    vals = np.array([c[key] for c in all_components])
    print(f"{key}: mean={vals.mean():.4f}  std={vals.std(ddof=1):.4f}")

Writing aggregate_seeds.py


In [9]:
!python aggregate_seeds.py

artifacts/idan_gate_seed42: {'sentiment-en-AU': 0.9158920879619135, 'sentiment-en-UK': 0.9481182795698925, 'sarcasm-en-AU': 0.7822475055307238, 'sarcasm-en-UK': 0.7228157980633229} -> ALTA=0.8194
artifacts/idan_gate_seed123: {'sentiment-en-AU': 0.9323767653127438, 'sentiment-en-UK': 0.9532423483808448, 'sarcasm-en-AU': 0.7573260073260073, 'sarcasm-en-UK': 0.739457079970653} -> ALTA=0.8359
artifacts/idan_gate_seed2024: {'sentiment-en-AU': 0.9190050649123829, 'sentiment-en-UK': 0.9531553398058252, 'sarcasm-en-AU': 0.7934161971365743, 'sarcasm-en-UK': 0.7442767758557232} -> ALTA=0.8316

=== Summary across seeds ===
ALTA score: mean=0.8290  std=0.0086  min=0.8194  max=0.8359
sentiment-en-AU: mean=0.9224  std=0.0088
sentiment-en-UK: mean=0.9515  std=0.0029
sarcasm-en-AU: mean=0.7777  std=0.0185
sarcasm-en-UK: mean=0.7355  std=0.0113
